# 2장 1강: 회귀와 분류 모델의 예측 — 실습문제

## 실습 목표

- 금액을 예측하는 회귀와 구매 여부를 예측하는 분류를 구분합니다.
- 분류의 점수, 예측 확률, 최종 클래스가 각각 무엇인지 구분합니다.
- 교안의 함수를 이용해 Sigmoid·Softmax 확률을 계산합니다.
- 임계값 비교와 Argmax로 최종 클래스를 선택합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 모든 숫자와 계산 방법은 제공된 「2장 1강: 회귀와 분류 모델의 예측」의 본문 예제와 연습문제를 바탕으로 구성했습니다. 주어진 가중치로 출력만 계산하며 모델을 학습시키는 실습은 없습니다.

## 실습 구성과 교안 대응

| 구분 | 내용 | 교안 근거 및 구성 방식 |
|---|---|---|
| 필수 1 / 문제 1-1 | 회귀 예측값과 분류 점수 | 1절 본문의 고객 3명 예제 |
| 필수 1 / 문제 1-2 | Sigmoid와 임계값 | 1절의 점수에 2·3절의 계산 규칙을 연결 |
| 필수 2 / 문제 2-1 | 한 상품의 Softmax와 Argmax | 2·3절 본문의 [2,1,0] 예제 |
| 필수 2 / 문제 2-2 | 고객별 Softmax와 Argmax | 2·3절 본문의 3행 점수 배열 예제 |
| 과제 / 문제 3-1 | 임계값 0.5와 0.7 비교 | 연습문제 3의 소문항 1·2를 나누어 연습 |
| 과제 / 문제 3-2 | 한 상품의 Softmax 계산 복습 | 연습문제 2의 소문항 3에 3절의 Argmax 적용 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 교안 본문 예제와 연습문제의 일부를 기초 실습 4문제·복습 과제 2문제로 구성했습니다. 새 공식을 유도하거나 여러 오류 유형을 한꺼번에 분류하는 과제는 요구하지 않습니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 위에서 아래로 실행합니다.
2. 표에서 숫자가 무엇을 뜻하는지 확인한 뒤 계산합니다.
3. 함수 코드는 문제에 제공된 교안의 형태를 그대로 사용해도 됩니다. `return` 앞의 들여쓰기를 유지하세요.
4. 확률을 소수점 아래 세 자리로 표시할 때만 `np.round()`를 씁니다. 임계값과 비교할 때는 반올림 전 확률을 사용합니다.
5. 각 문제의 함수와 변수를 다시 작성합니다. 한 벡터용 Softmax와 여러 행용 Softmax는 사용하는 코드가 다릅니다.
6. 해석 질문의 답은 계산값과 함께 쉬운 문장으로 적으면 됩니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 특성 | 계산에 넣는 정보. 여기서는 방문 횟수와 장바구니 추가 횟수 |
| 가중치 w / 편향 b | 입력에 각각 곱할 숫자 / 곱해서 더한 결과에 추가로 더할 숫자 |
| 회귀 | 금액처럼 ‘얼마인가’를 숫자로 예측하는 문제 |
| 분류 / 클래스 | 정해진 종류 중 하나를 고르는 문제 / 그 종류의 이름 |
| 이진 분류 | 두 종류 중 하나를 선택하는 분류. 여기서는 구매·미구매 |
| 다중 클래스 분류 | 세 개 이상의 종류 중 하나를 선택하는 분류. 여기서는 의류·식품·가전 |
| 점수 z / Logit(로짓) | 확률로 바꾸기 전 모델이 계산한 숫자 |
| 확률 p | 모델이 어떤 클래스에 부여한 가능성을 0~1의 형태로 나타낸 값. 0.7은 70% |
| 실제값 y / 예측값 ŷ | 실제로 관측된 정답 / 모델이 계산한 결과. ŷ는 ‘와이 햇’으로 읽음 |
| Sigmoid(시그모이드) | 이진 분류에서 점수 하나를 클래스 1의 확률로 바꾸는 함수 |
| Softmax(소프트맥스) | 한 데이터의 클래스별 점수를 합이 1인 확률들로 바꾸는 함수 |
| Threshold(임계값) | 확률을 0 또는 1로 결정할 때 비교하는 기준값 |
| Argmax(아그맥스) | 가장 큰 값이 있는 위치 번호를 찾는 연산 |
| 인덱스 | 배열의 위치 번호. Python에서는 첫 위치가 0 |
| 행 / 열 / Shape | 가로줄 / 세로줄 / 배열의 모양. (3,2)는 3행 2열 |

이 실습의 이진 분류에서 **0은 미구매, 1은 구매**입니다. 다중 클래스의 인덱스는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자 1이어도 이진 분류의 클래스 1과 다중 클래스의 인덱스 1은 서로 다른 이름을 가리킵니다.

####
1. 아핀 변환
- 선형 변환 결과에 일정한 벡터나 숫자 b를 더하는 계산
- 입력을 조합한 뒤에 기준 위치를 옮긴다.
- T(x) = xW + b
- 선형 결과 [6,6]에 b = [1, -2]를 더하면 [7,4]
- 왜 쓰는가? 화면 속 점 [3,2]를 가로, 세로가 n배 늘어났고 옮긴다는 가정하에 아핀 변환을 사용할 수 있다.
###
2. 뉴런
- 뉴런은 입력을 받아 출력 하나를 만드는 계산 단위
- Ex) z = x1w1 + x2w2 + b
- 학생의 [출석, 과제] 기록 [3,2]를 받아 중간 값(은닉층 출력 결과)를 만드는 계산기가 있다.
- 가중치 [1,2]를 곱해 3+4 =7을 만들고 편향 -4를 통해 3을 얻는다.
###
3. 레이어와 신경망
- 레이어는 여러 뉴런을 모아놓은 계산 단계
- 신경망은 이런 계산들을 연결한 모델
- 학생의 기록 [3,2]를 단계별 계산을 하면
- 첫 단계의 뉴런 두 개가 각각 3과 -3을 만들고 ReLU를 거쳐 3,0 을 만들어내면
- 두 번째 단계에서는 [3,0]을 이어받아서 3x2 + 0x(-1) + 1 = 7을 만든다.
- 학생 기록 -> 중간 숫자 -> 최종 점수 | 이 각 단계가 레이어이다.
###
4. 활성화 함수
- 활성화 함수는 뉴런의 가중합에 편향을 더한 값을 다시 바꾸는 규칙이다.
- Ex) z = xw + b를 구하고 난 뒤 a = f(z)를 계산
- 학습실 이용 시간을 4시간 초과한 경우 금액을 부과하기 위해 따로 기록을 남긴다고 가정하면
- z = -4를 계산, 3시간을 이용했다면 z = -1로, ReLU적용 시 0이 되어 추가 금액을 부과하지 않겠다.
- 이처럼 이미 계산된 식에 임의의 식을 더해서 그 기록과 규칙을 바꾸려고 하는 것이 활성화 함수이다.
###
5. 비선형 함수
- 비선형 함수는 선형 변환이 지켜야하는 덧셈 및 배수 관계를 모두 만족하지 않는 함수
- 추가 이용시간 y = ReLU(이용시간 -4)를 계산한다고 가정하면
- 3시간 이용자, 4시간 이용자는 추가 시간이 0인데, 5시간, 6시간 이용자는 1시간, 2시간이다.
- 이용 시간이 한 시간 늘어나도 기준 시간은 결과가 그대로이기 때문에 4시간 전까지 올바른 선형관계를 보이기 어렵다.
###
6. 은닉층
- 입력과 최종 출력 사이에서 중간 값을 만드는 계산 단계
- 은닉이라는 단어가 들어가 있지만 Print()로 중간값들을 확인할 수는 있음
- 학생의 출석 및 과제 기록이 [3,2]로 최종 점수를 바로 만들지 않고 중간 계산을 거쳐서 검토한다고 가정하면
- 두 뉴런으로 [3, -3]값을 만든 뒤 ReLU를 적용하여 [3,0]으로 변환한다.
- 이 과정을 진행하는 부분을 은닉층이라고 부른다.
- 출력층(최종 결과 산출)은 이 [3,0]을 받아서 최종 점수를 출력한다.
- 실제 신경망 계산에서는 입력을 여러 중간값으로 바꾸어 계산을 진행한다.
###
7. 출력층
- 출력층은 모델의 최종 결과를 산출하는 계산 단계이다.
- 앞 단계에서 학생의 중간 값 H = [5,3]을 얻었다고 가정해보면 이 각 컬럼에 대한 점수들을 가지고 실제 점수에 반영해보겠다.
- 10점 만점이라고 가정하고 가중치르 [2, -1], 편향을 1로 가정하면
- 5x2 + 3x(-1) +1 = 8점, 최종적으로 학생으 점수 8점을 출력한다.
- 출력층에 ReLU가 없을 때, 만약에 H = [0,3]이면 0 - 3 + 1 = -2로, 최종 출력값은 0이 아니라 -2로 계산될 수 있다. 
###
8. 순전파
- 순전파는 입력부터 최종 출력까지 계산을 앞 방향으로 진행하는 과정
- 학생 기록 [3,2]를 모델에 넣고 계산 결과를 순전파로 따라가보면 
- (1)가중합과 편향을 계산 Z1 = [3, -3]
- (2) ReLU를 적용하여 H = [3,0]을 만들고
- (3) 출력층에서 3x2 + 0x(-1) + 1 = 7을 산출한다.
- (1) -> (2) -> (3) 과정을 순전파 라고 부른다.
###
9. 분류 점수와 Logit
- 분류 점수(Logit)은 확률로 바꾸기 전에 모델이 계산한 숫자
- 메일이 스팸인지 분류하는 모델이 중간 점수 z=0.8이라는 점수를 냈다고 가정해보면
- 이 숫자 자체는 아직 80%를 의미하는 것이 아니라 그냥 그 점수 값을 의미한다.
- 이 숫자가 sigmoid라는 함수를 만나 0.69라는 확률로 계산이 될 수 있다.
- y = ax + b = 14
###
10. 시그모이드
- sigmoid 는 점수 하나를 0과 1 사이로 바꾸는 함수
- z= 0.8이면 -z = -0.8이고 e의 -0.8의 제곱은 약 0.449329이면
- 여기다가 1을 더한 1.449329로 1을 나누면 약 0.6899라는 결과를 얻을 수 있다.
###
11. 소프트맥스
- 소프트맥스는 한 대상의 여러 클래스 점수를, 합이 1인 클래스별 확률로 바꾸는 함수
- 예를 들어, 문의 한 건을 [배송, 환불, 기타] 중 하나로 분류해야 한다면
- 모델이 예측한 결과 점수가 [2,1,0]이라고 가정해보면
- 최대값 2를 빼서 [0, -1, -2]로 만들고 지수값 약 [1, 0.36, 0.13]을 구한다. -> 2를 왜 빼요?
- 각 지수 값의 합계인 약 1.5로 나누면 확률은 약 [0.66, 0.24, 0.09]이고, 이 셋의 점수 합은 1이다.
- 이렇게 클래스별로 확률을 만든 뒤 가장 큰 확률을 가진 0.66(배송)을 선택하겠다.
- 문의 종류마다 각 배송, 환불, 기타의 확률값을 모두 볼 수 있다.
###
12. Argmax
- 가장 큰 값이 있는 위치 번호를 찾는 연산
- 에를 들어 문의의 [배송, 환불, 기타] 확률이 [0.2, 0.6, 0.2]라고 가정해보면
- 가장 큰 값은 0.6이고 위치는 두 번째이다.
- 따라서 Python에서 표현하면 np.argmax(p) = 1 을 반환한다.
- 읽을 때에는 1번 환불이라고 읽으면 된다.
- 확률과 이름을 한번에 같이 표현할 때 쓴다.
### 
13. 조건부 확률
- 특정 조건을 만족하는 경우만 모아서 보는 확률
- P(구매 | 장바구니 추가) = 장바구니 추가한 고객 중에서 실제로 구매할 확률, | 뒤쪽이 조건이다.
- 장바구니 추가 고객 6명 중 4명이 구매했다면 관측 구매 비율은 4/6 = 0.66정도
- 단, 고객 10명 기준이 아니라 장바구니를 추가한 고객 6명을 기준(조건)으로 계산했다는 점이 중요하다.
###
14. 베르누이 분포
- 결과가 두 가지일 때, 한 번의 결과를 0또는 1로 나타내는 확률 모형
- 계산식 Y = 1에서 p, y = 0에서 확률은 1-p
- Kant 수업에서 한 학생이 과제를 제출했는지만 확인한다고 해보면 제출은 1, 미제출은 0으로 기록한다.
- 제출 확률을 p = 0.6으로 보면 제출에 0.6, 미제출에 0.4라는 점수를 부여한다.
- 이처럼 한 가지 결과를 두 가지로 표현하는 것이 베르누이 모형이다.
###
15. Likelihood(우도)
- 이미 관측한 데이터를 고정해놓고, 파라미터에 따라 그 데이터에 부여되는 확률을 비교하는 함수
- y = [1, 0, 1, 1, 0], p = 6이면
- L(0.6) = 0.6 x 0.4 x 0.6 x 0.6 x 0.4 = 0.03456
- 이 경우 L(p)는 p를 넣었을 떄의 우도라고 읽을 수 있다.
- 우의 계산식을 말로 풀어보면
- 다섯 개의 과제 제출 기록이 [1, 0, 1, 1, 0]이라고 하면 이 기록을 그대로 공통 제출 확률 두 개를 비교한다.
- p = 0.6이면 실제 기록에 준 확률의 곱은 0.6 x 0.4 x 0.6 x 0.6 x 0.4 = 0.03456이고
- p = 0.8이면 실제 기록에 준 확률의 곱은 0.8 x 0.2 x 0.8 x 0.8 x 0.2 = 0.002048이다.
- 제출 확률 80으로 설정을 해놓았다면 제출자에게는 유리하지만 미제출 두 명에게는 20%씩만 준다.
- 왜 쓰는가?: 어느 모델의 설정한 값이 지금 현재 기록에 더 잘 맞는지 숫자로 비교하고 싶다.
###
16. Probability와 Likelihood의 관점
- 두 개념은 확률식을 서로 다른 관점에서 바라본다
- Probability는 모델을 정해두고 가능한 결과를 살펴보는 방법이라면
- Likelihood는 실제 출력된 결과를 보고 모델의 파라미터 값을 보겠다는 의미
- 확률 관점에서 p = 0.6일 때, 이 0.6이라는 숫자값이 중요하다. 구매 확률이 60%인가? 라는 질문을 던진다면
- Likelihood 관점에서는 실제 구매, 미구매 기록을 고정한 상태에서 L(0.6), L(0.8)을 비교하겠다는 의미
- 질문: 수업 전에 이번 AI기초 수학 과제를 제출할 확률을 0.6으로 설정했다면 원우가 이번 과제를 제출할 확률은? <- 확률 관점
- 수업이 끝나고 난 뒤에 [1, 0, 1, 1, 0]이라는 실제 기록을 확인한 뒤 P=0.6과 P=0.8중 어느 설정이 이 기록에 더 큰 확률을 주는가? <- 우도 관점
###
17. 파라미터 후보와 후보 비교
- 파라미터 후보라는 것은 시험해 볼 파라미터 값을 의미한다.
- 각 후보에 같은 데이터를 넣어 우도를 구한 다음에 가장 큰 우도를 주는 후보를 선택한다.
- p = [0.2, 0.4, 0.6, 0.8]의 우도는 순서대로 [0.005, 0.023, 0.034, 0.020]이다.
- 그렇다면 가장 큰 우도로 연결된 후보는 p = 0.6이다.
- 이 계산식을 말로 풀어보면
- [1, 0, 1, 1, 0]에 대한 제출 확률을 p = [0.2, 0.4, 0.6, 0.8] 산출하면 우도는 [0.005, 0.023, 0.034, 0.020]이다.
- 따라서 제출 확률에 대한 우도 값이 가장 큰 것은 p = 0.6이기 때문에 지금 [1, 0, 1, 1, 0]기록에 더 잘맞는 설정값(p = 0.6)을 찾는 일이다.
- p가 높다고해서 우도 관점에서 무조건 좋은 것은 아니다.(p = 0.8 탈락)
###
18. 최대 우도 추정 Maximum Likelihood Estimation MLE
- 관측 데이터의 우도가 가장 커지는 파라미터를 찾는 방법
- 공통 p를 사용하는 독립 베르누이 모형에서는 MLE = 구매수 / 전체 고객 수
- 구매 3명, 전체 5명이면 3/5 = 0.6 | 이 모형에서는 0 ~ 1 사이에서 0.6이 우도를 가장 크게 만든다.
- 예를 들어 예약 방문 기록이 [1, 0, 1, 1, 0]이라면 방문 3명, 미방문 2명이 모든 예약에 같은 방문확률 P를 사용하겠다라고 가정하면
- 각 결과가 독립인 베르누이 모형에서는 우도를 가장 크게하는 P가 방문수 / 전체수인 3/5 = 0.6
- 따라서 0.6이 최대우도추정값이고 그 때 전체 기록의 우도는 0.034정도
- 사람이 임의로 정한 확률 대신에 관측 자료 결과에 근거하여 파라미터를 정하기 위해서 최대 우도 추정을 기준으로 사용한다.
###
19. 인덱스와 Argmax
- 인덱스는 배열의 위치 번호, np.argmax()는 가장 큰 값 자체가 아니라 그 값이 들어있는 위치 번호를 반환
- 우도 배열 [0.005, 0.023, 0.034, 0.020]이라고 가정할 때,
- 가장 큰 우도는 0.034이며 순서는 세 번째에 있다는 것을 알기 위함
- 가장 좋은 평가값을 빠르게 찾고 그 뒤 값을 만든 설정까지 빠르게 찾기 위함
###
20. 로그
- 어떤 밑을 몇 제곱하면 이 값이 되는가를 알려주는 연산자
- 예를 들어 2를 3제곱하면 8이니까 밑이 2인 8의 로그는 3이다.
- 그렇다면 우도랑 무슨 상관인가?
- 예약 세 건의 실제 결과에 확률이 0.8, 0.7, 0.7이라고 가정하면, 우도는 이 세값을 곱한 0.392값을 의미한다.
- 이 우도의 자연로그는 ln(0.392)로 구할 수 있고 ln(0.8), ln(0.7) + ln(0.7)로 얻을 수 있다.
- 즉 로그의 특성을 이용하면 확률을 모두 곱하는 과정을 확률의 로그를 더하는 과정으로 바꿀 수 있다.
- 기록이 매우 많아서 계속 곱해서 0에 무한히 가까워지면 구분하기 어려우므로, 같은 비교지만 로그를 써서 안정적으로 계산하겠다는 의미이다.
###
21. 로그우도 (Log Likelihood)
- 우도에 자연로그를 적용한 값
- 각 관측 확률의 자연로그를 더해서도 같은 값을 얻는다
- 같은 데이터에서는 큰 로그 우도를 선호한다.
- q = [0.8, 0.7, 0.7]이면 L = 0.392고, ln(L) = ln(0.8) + ln(0.7) + ln(0.7) = -0.936이다.
- 코드에서는 np.sum(np.log(q))를 사용할 수 있다.
- 예를 들어 다른 모델이 같은 세 결과에 각각 0.5를 주면 우도는 0.125
- 로그 우도는 약 -2.0794값을 갖는다.
- -0.9365 > -2.0794이기 때문에 첫 모델을 선택하려고 한다.
- 우도로 비교해도 0.392 > 0.125이므로 같은 선택을 한다.
###
22. 같은 데이터에서의 모델 비교
- 같은 고객의 같은 실제 결과에 대해 각 모델이 부여한 우도를 비교하겠다.
- 한 고객에게 더 좋은 확률을 준 무델이 다른 고객 또는 전체 데이터에서 항상 좋은것은 아니다.
- 실제값 [1, 0, 1]에서 A의 관측 확률 [0.8, 0.6, 0.6]은 우도 0.288
- B의 관측확률 [0.6, 0.8, 0.5]는 0.240 <- 이 두 모델을 비교했을 때는 A의 우도값이 높으니까 A 모델을 선택하겠다.
###
23. 손실함수와 Loss
- 모델의 예측을 실제 정답과 비교하여 숫자로 평가하는 규칙
- 금액 예측은 실제값과 예측값의 차이로 MSE를 구할 수 있다.
- 구매 여부와 상품 종류는 실제 정답에 주어진 확률로 BCE 또는 CE를 구할 수 있다.
- 문제 종류에 맞게 계산 규칙을 선택한다.
- 예를 들어, 배달 시간 예측 모델 두 개를 비교하겠다고 가정할 때, 실제 시간은 [10, 20, 30]분이고, 예측 시간은 [12, 18, 27]분이라면
- 각 주문에서 실제 시간과 예측시간의 차이를 제곱하면 [4,4,9]이다. <- 평균을 계산해보면 17 / 3 = 5.66
- 같은 주문에서 B의 평균 제곱오차가 8.667이라면 5.66인 A모델이 더 좋았다고 평가할 수 있다.
- 손실 함수는 모델의 예측을 비교하고 학습 중 어떤 것을 줄여야하는지 정하는 역할을 한다.
###
24. 제곱오차합과 SSE(Sum of Squared Errors)
- 제곱 오차를 모두 더한 값
- 제곱 오차 [4, 4, 9]의 SSE는 17이다.
- 배달 3건에 대해서 A 모델의 제곱 오차가 [4,4,9], B의 제곱오차가 [1,16,9]라고 가정해보면
- A의 합계는 17, B는 26 <- 이 합계가 SSE이다.
- 같은 세 건에 대해서는 A의 전체 제곱오차가 더 작다.
- B가 비록 첫 주문을 A모델보다 더 잘 맞췄더라도 전체 합계에서는 A가 더 좋은 모델일 수 있다.
- SSE를 전체 주문 건수로 나누면 MSE가 된다.
- 이걸 하는 이유: SSE를 통해 MSE를 구할 수 있고, 합계가 커진 이유가 오차 그 자체 때문인지, 데이터의 수에 영향을 받았는지 구분할 수 있다.
###
25. 평균제곱오차 MSE(Mean Squared Error)
- 제곱 오차의 평균
- 각 고객의 오차를 제곱하고 모두 더한 뒤 전체 고객 수로 나눈다.
- 배달 세 건의 실제 시간이 [10, 20, 30]분이고 예측이 [12, 18, 27]분이면 실제 예측은 [-2, 2, 3]이고 
- 각 오차의 제곱은 [4,4,9] 합계 17을 전체 3(전체 고객 수)으로 나누면 5.66
- 다른 B모델의 MSE가 8.66이라면 A모델을 선택하는 것이 맞다.
- 왜 중요한가?: 여러 수치의 예측 제곱오차를 하나의 평균으로 요약해 비교하기 위해서.
- 오차를 전체 수로 평균을 내기 때문에 각 한건에 대한 오차 범위를 대략적으로 파악할 수 있다.
###
26. MSE의 단위
- 오차를 제곱하면 단위도 제곱하면 된다.
- 단위가 만원이었다면 제곱오차에 대한 해석도 (만원)² 단위로 읽으면 된다.
###
27. 이진 교차 엔트로피와 BCE
- Binary Cross Entropy: 구매, 미구매 두가지 결과를 구분하는 분류 손실함수
- 한 고객의 BCE는 실제 정답 확률에 음의 로그를 적용한 값
- 예를 들어보면 
- 두 예약자의 실제 방문 결과가 [1,0]이고, A모델이 예측한 두 예약자의 방문 확률이 [0.8, 0.3]이라고 가정하면
- 첫 번째 손님의 손실은 -ln(0.8) = 약 0.223, 둘째는 미방문이니까 -ln(0.7) = 약 0.356정도
- B가 두 사람 모두에게 방문 확률 0.5를 주었다면 두 손실은 각각 약 0.6931로 평균도 0.6931로 나타난다.
- 같은 두 예약 확률에 대해서는 BCE가 작은 A모델을 선택한다.
- 왜 쓰는가?: 두 예측 결과를 학습하고 평가할 기준으로 쓴다. 각 정답에 어느정도의 확률을 부여했는지 알 수 있다.
###
28. 범주형 교차 엔트로피와 CE
- Categorical Corss Entropy, 여러 종류 중 실제 정답이 하나인 분류에 쓰이는 손실
- 상품 B의 정답이 수박 [배, 수박, 사과] 확률이 [0.2, 0.6, 0.2]라면 CE = -ln(0.6) = 약 0.511
- 예를 들어 과일 종류를 [배, 수박, 사과]순서로 분류한다고 했을 때, 
- 첫 문의의 실제 정답은 수박이고 예측 확률이 [0.2, 0.6, 0.2]일 때 정답인 수박의 0.6을 선택하여 -ln(0.6) = 약 0.511을 계산한다.
- 두번째 정답 확률이 0.8이라면 CE = 0.2231, 두 과일 선택의 평균을 내보면 0.367 정도
- 왜 쓰는가?: 과일 선택처럼 대상마다 열의 확률을 고르고 대상별 손실을 평균 내는 계산을 활용할 수 있다.
###
29. 예측 클래스와 손실의 차이
- 예측 클래스는 모델이 최종적으로 선택한 종류
- 손실은 실제 정답에 준 확률을 평가한 값
- 정답을 맞추면 손실이 0이냐? -> 아님
- 예를 들어, 실제 구매 고객에게 확률 0.9와 0.6을 주면 임계값 0.5에서 둘 다 구매로 예측
- 하지만 BCE는 약 0.105와 0.511로 다를 수 있음
- 즉, 정답을 맞췄다는 사실만으로 예측의 품질을 알 수가 없기 때문에 손실 점수를 확인해야한다.
- 두 가지 사실, 정답을 맞췄는가? / 확률을 얼마나 정확하게 맞췄는가?에 대한 답을 할 수 있다.
###
30. 정규분포(가우시안 분포)
- 평균을 중심으로 좌우가 대칭인 종 모양의 연속형 분포
- 예를 들면 배달 모델에서 주문 도착 시간을 20분으로 예측했다고 가정하면
- 실제 배달 시간이 19분, 21분처럼 약간의 오차가 발생할 수는 있다.
- 비슷한 조건에서 주문했을 때, 작은 차이는 자주, 큰 차이는 드물게 나타남을 표현하기 위해 정규분포를 가정할 수 있다.


## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이며, `np`는 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 1. 회귀 예측과 이진 분류 이해하기

### 문제 1-1. 같은 입력으로 회귀 예측값과 분류 점수 계산하기

#### 문제 설명

온라인 쇼핑몰 고객의 활동 데이터입니다. 교안 1절의 예시를 사용합니다.

| 고객 | 방문 횟수 | 장바구니 추가 횟수 | 실제 구매 금액(원) | 실제 구매 여부 |
|---|---:|---:|---:|---:|
| A | 4 | 2 | 12000 | 1 |
| B | 2 | 0 | 0 | 0 |
| C | 6 | 3 | 24000 | 1 |

입력 X에는 방문 횟수와 장바구니 추가 횟수만 순서대로 넣습니다. 행 순서는 A, B, C입니다. 구매 여부의 0은 미구매, 1은 구매를 뜻합니다.

| 예측 대상 | 가중치 | 편향 | 계산식 |
|---|---|---:|---|
| 구매 금액 | w_reg = [1000, 3000] | b_reg = 5000 | y_reg = X @ w_reg + b_reg |
| 구매 여부의 점수 | w_clf = [0.3, 0.8] | b_clf = −2 | z = X @ w_clf + b_clf |

가중치와 편향은 계산 원리를 확인하기 위해 주어진 값입니다. 이 문제에서는 분류 점수까지만 계산합니다.

#### 계산식을 말로 읽기

`X @ w + b`는 ‘고객 한 명의 첫 입력×첫 가중치 + 둘째 입력×둘째 가중치 + 편향’입니다. `@`는 이러한 곱셈과 덧셈을 고객별로 계산합니다. 이번에는 두 입력을 사용하므로 가중치도 두 개입니다.

#### 요구사항

1. 입력 행렬 `X`와 두 모델의 가중치·편향을 만들고, `X.shape`를 출력하세요.
2. 교안의 `@`와 `+`를 사용하여 세 고객의 예측 구매 금액 `y_reg`와 분류 점수 `z`를 계산하세요. 두 결과와 Shape를 출력하세요.
3. 고객 A의 금액 예측 오차를 “예측 구매 금액 − 실제 구매 금액”으로 계산하고 출력하세요.
4. 고객 A의 두 출력값을 수식에 숫자를 대입하여 확인하고, Q1~Q5에 답하세요.

#### 작성 도움말

`X`는 `np.array([[4, 2], [2, 0], [6, 3]])`으로 만듭니다. `y_reg[0]`은 고객 A의 예측 구매 금액입니다. 실제 금액과 실제 구매 여부는 입력 특성에 넣지 않습니다.

#### 해석 질문

**Q1.** 구매 금액 예측과 구매 여부 예측은 각각 어떤 문제 유형인가요?

**Q2.** X, y_reg, z의 Shape는 무엇이며, 결과 배열의 첫 번째 값은 누구의 결과인가요?

**Q3.** 세 고객의 예측 구매 금액은 각각 얼마인가요? 고객 A의 오차는 무엇을 뜻하나요?

**Q4.** 세 고객의 분류 점수는 각각 얼마인가요? A의 점수 0.8을 구매 확률 80%로 읽어도 되나요?

**Q5.** 주어진 코드로 예측값을 계산하면 가중치와 편향도 자동으로 학습되나요?

#### 제출 결과

- 두 모델의 계산 코드, 출력값과 Shape
- 고객 A의 대입 계산 과정과 금액 예측 오차
- Q1~Q5 답변

In [3]:
# 문제 1-1. 같은 입력으로 회귀 예측값과 분류 점수 계산하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.
X = np.array([[4,2], [2,0], [6,3]])
w_reg = np.array([1000, 3000])
b_reg = 5000
w_clf = np.array([0.3, 0.8])
b_clf = -2

y_reg = X @ w_reg + b_reg
z = X @ w_clf + b_clf
error_a = y_reg[0] - 12000

print("X shape:", X.shape)
print("y_reg shape:", y_reg.shape)
print("z shape:", z.shape)
print("error_a:", error_a)


X shape: (3, 2)
y_reg shape: (3,)
z shape: (3,)
error_a: 3000


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 구매 금액 예측과 구매 여부 예측은 각각 어떤 문제 유형인가요?**

- 답변: 구매 금액 예측은 '얼마를 구매하는가?'를 숫자로 표현하는 회귀 문제
- 구매 여부 예측은 미구매(0)와 구매(1)를 구별하는 이진 분류

**Q2. X, y_reg, z의 Shape는 무엇이며, 결과 배열의 첫 번째 값은 누구의 결과인가요?**

- 답변: X는 고객 3명과 입력 2개씩(장바구니, 방문횟수) 담으므로 (3,2)
- y_reg와 z는 각각 고객당 값 하나씩, 숫자 세 개를 담은 1차원 배열(3,)
- 입력과 결과의 순서를 고려하여 A -> B -> C 배열로 순서대로 해석

**Q3. 세 고객의 예측 구매 금액은 각각 얼마인가요? 고객 A의 오차는 무엇을 뜻하나요?**

- 답변: A는 4x0.3 + 2x0.8 -2 = 0.8
- B는 2x0.3 + 0x0.8 -2 = -1.4
- C는 6x0.3 + 3x0.8 -2 = 2.2
- 아직 확률 계산은 들어가지 않았기 때문에 0.8을 80%라고 해석하지 않는다. 

**Q4. 세 고객의 분류 점수는 각각 얼마인가요? A의 점수 0.8을 구매 확률 80%로 읽어도 되나요?**

- 답변: A는 4x0.3 + 2x0.8 -2 = 0.8
- B는 2x0.3 + 0x0.8 -2 = -1.4
- C는 6x0.3 + 3x0.8 -2 = 2.2
- 점수는 음수나 1보다 큰 값도 가능하다(확률이 아니므로). Sigmoid를 통해 확률로 변환할 수 있다.

**Q5. 주어진 코드로 예측값을 계산하면 가중치와 편향도 자동으로 학습되나요?**

- 답변: 아니오. w_reg.b_reg.w_clf.b_clf는 처음에 주어진 그 값 그대로 사용
- 오차를 출력하는 것만으로 가중치나 편향이 바뀌지는 않는다.

### 문제 1-2. Sigmoid 확률과 Threshold로 구매 여부 결정하기

#### 문제 설명

문제 1-1의 고객별 분류 점수를 사용합니다. 계산의 출발점을 분명히 하기 위해 점수를 다시 배열로 만드세요.

| 고객 | 점수 z |
|---|---:|
| A | 0.8 |
| B | −1.4 |
| C | 2.2 |

교안의 공식 `Sigmoid(z) = 1 / (1 + e^(−z))`로 구매 확률 p를 계산합니다. e는 약 2.718인 자연상수이며, `np.exp(-z)`가 e^(−z)를 계산합니다. 미구매 확률은 `1 − p`입니다.

결정 규칙은 **p가 임계값 이상이면 구매(1), 작으면 미구매(0)**입니다. 임계값은 0.5와 0.7을 각각 적용합니다. 추가로 점수가 정확히 0인 경우도 확인합니다.

#### 수식과 기호 읽기

`1 / (1 + e^(−z))`는 **z의 부호를 바꾼다 → e의 그 제곱값을 구한다 → 1을 더한다 → 그 전체로 1을 나눈다**는 순서입니다. 분수의 위 숫자는 분자, 아래 숫자는 분모입니다. 여기서는 분자가 1, 분모가 `1+e^(−z)`입니다.

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

함수는 위 코드를 그대로 사용해도 됩니다. z가 −1.4이면 −z는 +1.4입니다. 음수 앞에 다시 마이너스를 붙이면 양수가 됩니다.

#### 요구사항

1. 교안처럼 `def sigmoid(z):`와 `return 1 / (1 + np.exp(-z))`로 함수를 작성하세요.
2. 점수 배열 `[0.8, -1.4, 2.2]`에 Sigmoid를 적용하여 구매 확률 `p`와 미구매 확률 `1 - p`를 출력하세요.
3. `(p >= 임계값).astype(int)`로 임계값 0.5와 0.7의 예측을 각각 계산하고 출력하세요.
4. Sigmoid(0)을 계산하고, 이 확률을 임계값 0.5와 비교한 예측 클래스도 출력하세요.
5. 확률 표시는 소수점 아래 세 자리로 반올림하되, 임계값 비교에는 반올림 전 p를 사용하세요. Q1~Q5에 답하세요.

#### 작성 도움말

함수의 `return` 앞에는 들여쓰기를 넣습니다. 비교 결과 True와 False를 `.astype(int)`로 바꾸면 1과 0이 됩니다. `>=`에는 기준값과 같은 경우도 포함됩니다.

#### 해석 질문

**Q1.** A의 점수 0.8은 Sigmoid를 거치면 약 얼마의 구매 확률이 되나요? 미구매 확률은 얼마인가요?

**Q2.** B와 C의 구매 확률은 각각 얼마이며, 점수의 크기와 확률의 크기는 어떤 관계인가요?

**Q3.** 임계값 0.5와 0.7에서 예측 클래스는 각각 무엇인가요? 바뀌는 고객은 누구인가요?

**Q4.** 점수 0의 구매 확률은 왜 0.5인가요? 임계값 0.5에서 최종 클래스가 1인 이유는 무엇인가요?

**Q5.** Sigmoid 결과가 0과 1 사이에 있다는 사실만으로 예측 확률이 실제 구매 가능성을 정확히 나타낸다고 보장할 수 있나요?

#### 제출 결과

- Sigmoid 함수, 구매·미구매 확률, 두 임계값의 예측 클래스
- Sigmoid(0)의 계산 과정과 예측 클래스
- Q1~Q5 답변

In [5]:
# 문제 1-2. Sigmoid 확률과 Threshold로 구매 여부 결정하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.array([0.8, -1.4, 2.2])
p = sigmoid(z)
no_purchase = 1 - p
pred_05 = (p >= 0.5).astype(int)
pred_07 = (p >= 0.7).astype(int)
p_zero = sigmoid(0)
pred_zero = (p_zero >= 0.5).astype(int)

print("구매 확률:", np.round(p, 3))
print("미구매 확률:", np.round(no_purchase, 3))
print("임계값0.5 예측:", np.round(pred_05, 3))

구매 확률: [0.69  0.198 0.9  ]
미구매 확률: [0.31  0.802 0.1  ]
임계값0.5 예측: [1 0 1]


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. A의 점수 0.8은 Sigmoid를 거치면 약 얼마의 구매 확률이 되나요? 미구매 확률은 얼마인가요?**

- 답변: 구매 확률은 약 0.69, 즉 69% 확률로 구매하며 미구매 확률은 1 - 0.69 = 0.31이므로 31%
- 원래 점수인 0.8을 80%라고 해석하는 것은 적절하지 않다.
- Sigmoid(0.8) = 1 / (1 + e^(-0.8)) = 약 0.6899

**Q2. B와 C의 구매 확률은 각각 얼마이며, 점수의 크기와 확률의 크기는 어떤 관계인가요?**

- 답변: B는 0.198, C는 0.90 정도의 확률이다.
- 점수가 크다는 것은 sigmoid를 적용한 값인 구매 확률도 커진다는 것을 의미한다. 

**Q3. 임계값 0.5와 0.7에서 예측 클래스는 각각 무엇인가요? 바뀌는 고객은 누구인가요?**

- 답변: 임계값 0.5에서 [1, 0, 1], 임계값 0.7로 바꾸면 [0, 0, 1]로 바뀐다.
- A는 0.69로 0.7 미만이기 때문에 구매에서 미구매로 바뀐다.

**Q4. 점수 0의 구매 확률은 왜 0.5인가요? 임계값 0.5에서 최종 클래스가 1인 이유는 무엇인가요?**

- 답변: Z = 0이면 e^0 = 1이 되므로, sigmoid값이 1/2 = 0.5가 되므로, 기준 임계값 0.5에서는 최종 클래스 1로 분류될 수 있다.

**Q5. Sigmoid 결과가 0과 1 사이에 있다는 사실만으로 예측 확률이 실제 구매 가능성을 정확히 나타낸다고 보장할 수 있나요?**

- 답변: 아니오. 보장할 수는 없다. 하지만 Sigmoid가 만든 값은 모델이 부여한 예측 확률이다. 
- 확률이기 때문에 단정할 수는 없으며 실제 데이터와 비교하면서 예측이 적절한지 확인해야한다.

---

## 필수 2. Softmax와 Argmax 이해하기

### 문제 2-1. 한 상품의 Softmax 확률과 클래스 구하기

#### 문제 설명

교안 2·3절의 예제입니다. 한 상품을 의류·식품·가전 중 하나로 분류합니다.

| 클래스 | 의류 | 식품 | 가전 |
|---|---:|---:|---:|
| 인덱스 | 0 | 1 | 2 |
| 점수 z | 2 | 1 | 0 |

Softmax는 **각 점수의 지수값을 구하고 → 그 지수값들을 더하고 → 각 지수값을 합계로 나누는 계산**입니다. 수식에서 Σ(시그마)는 모두 더하라는 뜻입니다.

`Softmax(zᵢ) = e^(zᵢ) / Σ e^(zⱼ)`

i는 지금 확률을 구하려는 클래스의 위치, j는 합계에 넣는 모든 클래스의 위치입니다. 아래 코드는 교안처럼 가장 큰 점수를 먼저 빼고 계산합니다. 모든 점수에서 같은 값을 빼도 Softmax 확률은 유지됩니다.

```python
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)
```

함수는 그대로 사용하세요. 이번 문제에서는 이 성질을 증명하지 않고 계산 순서를 따라갑니다.

#### 요구사항

1. `z = np.array([2, 1, 0])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
2. 위 함수를 작성하고 `p = softmax(z)`를 계산하세요.
3. `np.round(p, 3)`과 `np.sum(p)`로 확률과 확률 합을 출력하세요.
4. `index = np.argmax(p)`로 인덱스를 구하고 index와 `classes[index]`를 출력하세요.

#### 작성 도움말

최댓값 2를 각 점수에서 빼면 [0,−1,−2]입니다. 지수값은 약 [1,0.3679,0.1353]이고 합은 약 1.5032입니다. 각 지수값을 이 합으로 나누세요. `np.argmax(p)`는 가장 큰 확률이 있는 위치를 반환합니다.

#### 해석 질문

**Q1.** 최댓값을 뺀 배열과 그 배열의 지수값은 각각 무엇인가요?

**Q2.** 세 클래스의 확률은 각각 얼마이며, 확률 합은 얼마인가요?

**Q3.** Argmax 결과와 예측 클래스 이름은 무엇인가요? Argmax가 확률값을 반환하나요?

**Q4.** 이 문제에서 Softmax를 사용하는 이유는 무엇인가요?

#### 제출 결과

- Softmax 함수와 확률·확률 합
- 예측 인덱스·클래스 이름 및 Q1~Q4 답변

In [6]:
# 문제 2-1. 한 상품의 Softmax 확률과 클래스 구하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.
# 1. `z = np.array([2, 1, 0])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
# 2. 위 함수를 작성하고 `p = softmax(z)`를 계산하세요.
# 3. `np.round(p, 3)`과 `np.sum(p)`로 확률과 확률 합을 출력하세요.
# 4. `index = np.argmax(p)`로 인덱스를 구하고 index와 `classes[index]`를 출력하세요.

import numpy as np

# 1. 점수 z와 클래스 이름 배열 classes 생성
z = np.array([2, 1, 0])
classes = np.array(["의류", "식품", "가전"])

# 2. Softmax 함수 정의 및 확률 p 계산
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

p = softmax(z)

# (참고) Q1 답변 확인용 중간 과정 출력
shifted_z = z - np.max(z)
exp_z = np.exp(shifted_z)
print("[참고] 최댓값을 뺀 배열:", shifted_z)
print("[참고] 지수값 배열 (반올림):", np.round(exp_z, 4))
print("-" * 40)

# 3. 확률(소수점 셋째 자리 반올림)과 확률 합 출력
print("3. Softmax 확률 및 합계:")
print("클래스별 확률 p (반올림):", np.round(p, 3))
print("확률의 합 np.sum(p):", np.sum(p))
print("-" * 40)

# 4. np.argmax(p)로 인덱스 구하고 index와 예측 클래스 이름 출력
index = np.argmax(p)
print("4. 최종 예측 결과:")
print("예측 인덱스 (index):", index)
print("예측 클래스 이름:", classes[index])

[참고] 최댓값을 뺀 배열: [ 0 -1 -2]
[참고] 지수값 배열 (반올림): [1.     0.3679 0.1353]
----------------------------------------
3. Softmax 확률 및 합계:
클래스별 확률 p (반올림): [0.665 0.245 0.09 ]
확률의 합 np.sum(p): 0.9999999999999999
----------------------------------------
4. 최종 예측 결과:
예측 인덱스 (index): 0
예측 클래스 이름: 의류


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 최댓값을 뺀 배열과 그 배열의 지수값은 각각 무엇인가요?**

- 답변: 최댓값 2를 뺸 배열 [0, -1, -2]
- 지수값은 [e^0, e^-1, e^-2] = [1, 0.36, 0.13]

**Q2. 세 클래스의 확률은 각각 얼마이며, 확률 합은 얼마인가요?**

- 답변: 공통 분모인 지수의 합은 1.503정도, 의류는 1 / 1.503이므로 약 0.66 정도
- 식품은 0.36 / 1.503이므로 약 0.24, 가전은 0.13 / 1.503이므로 약 0.09 정도
- 세 확률의 합은 계산 오차 범위 내에서 1이다.

**Q3. Argmax 결과와 예측 클래스 이름은 무엇인가요? Argmax가 확률값을 반환하나요?**

- 답변: Argmax는 0이고 예측 클래스는 의류, 가장 큰 확률 약 0.665가 첫 위치에 있으므로 인덱스가 0이다.

**Q4. 이 문제에서 Softmax를 사용하는 이유는 무엇인가요?**

- 답변: 상품을 의류 / 식품/ 가전으로 분류하는 다중 분류 문제이므로 SoftMax를 사용한다.

### 문제 2-2. 여러 고객의 확률과 클래스를 행별로 구하기

#### 문제 설명

교안 2·3절의 고객별 점수 예제입니다. 가로 한 줄인 행은 고객, 세로 한 줄인 열은 클래스입니다.

```python
Z = np.array([[2, 1, 0], [0, 2, 1], [1, 0, 3]])
classes = np.array(["의류", "식품", "가전"])
```

첫 행은 A, 둘째 행은 B, 셋째 행은 C입니다. 클래스 순서는 의류·식품·가전입니다. 한 고객의 세 점수끼리 Softmax를 계산합니다.

아래 교안의 함수를 **그대로 사용**하세요. 여러 행을 계산하는 데 필요한 옵션은 해설에서 다시 풀이합니다.

```python
def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
```

`axis=1`은 각 행 안의 세 클래스 값을 대상으로 계산하라는 뜻입니다. `keepdims=True`는 고객별로 구한 값이 자신의 행에 적용되도록 결과의 모양을 유지합니다.

#### 요구사항

1. 위의 Z와 classes를 만들고 Softmax 함수를 작성하세요.
2. `p = softmax(Z)`를 계산하고 `np.round(p, 3)`으로 확률을 출력하세요.
3. `Z.shape`, `p.shape`, `np.sum(p, axis=1)`을 출력하세요.
4. `predicted_indices = np.argmax(p, axis=1)`를 계산하고, 인덱스와 `classes[predicted_indices]`를 출력하세요.

#### 작성 도움말

한 행만 먼저 읽어 보세요. 첫 행 [2,1,0]은 앞 문제와 같으므로 같은 확률이 나옵니다. 고객별로 가장 큰 확률이 의류·식품·가전 중 어느 열에 있는지 찾습니다. 함수 안의 axis와 keepdims는 제시된 코드대로 사용하면 됩니다.

#### 해석 질문

**Q1.** Z와 p의 Shape는 무엇이며, 행과 열은 각각 무엇인가요?

**Q2.** 세 고객의 확률 분포와 각 고객의 확률 합은 무엇인가요?

**Q3.** 고객 A·B·C의 예측 인덱스와 클래스 이름은 각각 무엇인가요?

**Q4.** 이 문제에서 axis=1을 지정하는 이유는 무엇인가요?

#### 제출 결과

- 고객별 확률, Z·p의 Shape와 고객별 확률 합
- 고객별 예측 인덱스·이름 및 Q1~Q4 답변

In [7]:
# 문제 2-2. 여러 고객의 확률과 클래스를 행별로 구하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.
# 1. 위의 Z와 classes를 만들고 Softmax 함수를 작성하세요.
# 2. `p = softmax(Z)`를 계산하고 `np.round(p, 3)`으로 확률을 출력하세요.
# 3. `Z.shape`, `p.shape`, `np.sum(p, axis=1)`을 출력하세요.
# 4. `predicted_indices = np.argmax(p, axis=1)`를 계산하고, 인덱스와 `classes[predicted_indices]`를 출력하세요.

import numpy as np

# 1. Z와 classes 생성 및 행별 Softmax 함수 작성
Z = np.array([
    [2, 1, 0],  # 고객 A
    [0, 2, 1],  # 고객 B
    [1, 0, 3]   # 고객 C
])
classes = np.array(["의류", "식품", "가전"])

def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# 2. p = softmax(Z)를 계산하고 소수점 셋째 자리까지 반올림하여 출력
p = softmax(Z)
print("2. 고객별 Softmax 확률 p (반올림):\n", np.round(p, 3))
print("-" * 40)

# 3. Z.shape, p.shape, np.sum(p, axis=1) 출력
print("3. Shape 및 고객별 확률 합 확인:")
print("Z의 Shape:", Z.shape)
print("p의 Shape:", p.shape)
print("고객별 확률 합 np.sum(p, axis=1):", np.sum(p, axis=1))
print("-" * 40)

# 4. predicted_indices를 계산하고 인덱스와 예측 클래스 이름 출력
predicted_indices = np.argmax(p, axis=1)
print("4. 고객별 예측 결과:")
print("예측 인덱스 (predicted_indices):", predicted_indices)
print("예측 클래스 이름:", classes[predicted_indices])

2. 고객별 Softmax 확률 p (반올림):
 [[0.665 0.245 0.09 ]
 [0.09  0.665 0.245]
 [0.114 0.042 0.844]]
----------------------------------------
3. Shape 및 고객별 확률 합 확인:
Z의 Shape: (3, 3)
p의 Shape: (3, 3)
고객별 확률 합 np.sum(p, axis=1): [1. 1. 1.]
----------------------------------------
4. 고객별 예측 결과:
예측 인덱스 (predicted_indices): [0 1 2]
예측 클래스 이름: ['의류' '식품' '가전']


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. Z와 p의 Shape는 무엇이며, 행과 열은 각각 무엇인가요?**

- 답변: 둘 다 (3,3)이다.
- 행은 고객 A, B, C를 뜻하며, 열은 의류, 식품, 가전을 의미한다.
- Z에는 확률 변환 전 점수가 표현되며 P에는 변환 후 확률이 표현된다.
- 고객과 클래스의 순서는 바뀌지 않는다.

**Q2. 세 고객의 확률 분포와 각 고객의 확률 합은 무엇인가요?**

- 답변: 각 고객의 확률 합은 1이다.

**Q3. 고객 A·B·C의 예측 인덱스와 클래스 이름은 각각 무엇인가요?**

- 답변: 인덱스: [0, 1, 2] | 클래스명 ['의류', '식품', '가전']
- Argmax하면 확률이 가장 큰 순서대로 인덱스 지정됨

**Q4. 이 문제에서 axis=1을 지정하는 이유는 무엇인가요?**

- 답변: 한 고객의 세 클래스 점수를 함께 게산, 세 확률 중 최대값을 골라야하기 때문
- axis = 1 은 한 행의 값들을 대상으로 계산하겠다는 의미
- 행이 아닌 모든 고객을 대상으로 지수값을 계산하면 고객별 확률이 1이 되지 않음

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 같은 확률에 두 임계값 적용하기

#### 문제 설명

교안 연습문제 3의 소문항 1·2를 사용합니다. 이미 구매 확률이 주어져 있으므로 Sigmoid를 다시 적용하지 않습니다.

| 고객 | A | B | C | D |
|---|---:|---:|---:|---:|
| 구매 확률 | 0.85 | 0.40 | 0.65 | 0.20 |

규칙은 **확률이 임계값 이상이면 구매(1), 작으면 미구매(0)**입니다. 같은 확률에 임계값 0.5와 0.7을 각각 적용하세요. 실제 구매 여부는 주어지지 않았습니다.

#### 요구사항

1. `p = np.array([0.85, 0.40, 0.65, 0.20])`를 만드세요.
2. `pred_05 = (p >= 0.5).astype(int)`를 계산하고 출력하세요.
3. `pred_07 = (p >= 0.7).astype(int)`를 계산하고 출력하세요.
4. 답변 작성란에 고객별 두 예측값을 적고, 바뀐 고객을 찾으세요.

#### 작성 도움말

문제 1-2의 비교식을 그대로 사용합니다. 예를 들어 고객 C의 0.65는 0.5 이상이지만 0.7보다 작습니다. 임계값을 바꾸어도 주어진 확률은 그대로입니다.

#### 해석 질문

**Q1.** 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q2.** 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?

**Q3.** 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?

**Q4.** 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?

#### 제출 결과

- 두 임계값의 예측 배열
- 고객별 비교와 Q1~Q4 답변

In [8]:
# 문제 3-1. 같은 확률에 두 임계값 적용하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

# 1. `p = np.array([0.85, 0.40, 0.65, 0.20])`를 만드세요.
# 2. `pred_05 = (p >= 0.5).astype(int)`를 계산하고 출력하세요.
# 3. `pred_07 = (p >= 0.7).astype(int)`를 계산하고 출력하세요.
# 4. 답변 작성란에 고객별 두 예측값을 적고, 바뀐 고객을 찾으세요.

# 1. 고객별 구매 확률 배열 p 생성
p = np.array([0.85, 0.40, 0.65, 0.20])
print("1. 고객별 구매 확률 (p):", p)
print("-" * 45)

# 2. 임계값 0.5 적용 결과 계산 및 출력
pred_05 = (p >= 0.5).astype(int)
print("2. 임계값 0.5 예측 배열 (pred_05):", pred_05)
print("-" * 45)

# 3. 임계값 0.7 적용 결과 계산 및 출력
pred_07 = (p >= 0.7).astype(int)
print("3. 임계값 0.7 예측 배열 (pred_07):", pred_07)

1. 고객별 구매 확률 (p): [0.85 0.4  0.65 0.2 ]
---------------------------------------------
2. 임계값 0.5 예측 배열 (pred_05): [1 0 1 0]
---------------------------------------------
3. 임계값 0.7 예측 배열 (pred_07): [1 0 0 0]


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 임계값 0.5에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: 순서대로 1, 0, 1, 0

**Q2. 임계값 0.7에서 고객 A·B·C·D의 예측은 각각 무엇인가요?**

- 답변: 순서대로 1, 0, 0, 0

**Q3. 어느 고객의 예측이 바뀌었으며, 그 이유는 무엇인가요?**

- 답변: 고객C. 고객 C의 예측 구매 확률은 0.65였는데, 임계값이 0.7로 바뀌면 임계값보다 예측 구매 확률이 낮아져, 미구매로 예측한다.

**Q4. 임계값을 바꾸면 구매 확률 자체도 바뀌나요? 이 정보만으로 어느 기준이 더 정확한지 알 수 있나요?**

- 답변: 아니오. 구매 확률은 바뀌지 않는다. 실제 관측 데이터가 없으므로 이 정보만으로 어느 기준이 더 정확한지도 알 수 없다.

### 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기

#### 문제 설명

교안 연습문제 2의 소문항 3에 있는 점수입니다. 문제 2-1과 같은 방법으로 계산하세요.

| 클래스 | 의류 | 식품 | 가전 |
|---|---:|---:|---:|
| 인덱스 | 0 | 1 | 2 |
| 점수 | 1 | 2 | 1 |

이 한 상품은 세 종류 중 하나로 분류합니다. 교안 3절의 Argmax를 사용해 가장 높은 확률의 클래스도 확인하세요.

#### 요구사항

1. 문제 2-1의 한 벡터용 Softmax 함수를 그대로 작성하세요.
2. `z = np.array([1, 2, 1])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
3. `p = softmax(z)`를 계산하고 `np.round(p, 3)`과 `np.sum(p)`를 출력하세요.
4. `index = np.argmax(p)`를 계산하고 index와 `classes[index]`를 출력하세요.

#### 작성 도움말

한 벡터용 함수에는 axis=1을 쓰지 않습니다. 가장 큰 점수 2를 빼면 [−1,0,−1]이고, 지수값은 약 [0.3679,1,0.3679]입니다. 이 세 값을 더한 뒤 각각을 같은 합으로 나누세요.

#### 해석 질문

**Q1.** 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?

**Q2.** 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?

**Q3.** Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?

**Q4.** 의류와 가전의 확률이 같은 이유는 무엇인가요?

#### 제출 결과

- 한 벡터용 Softmax 함수, 확률·확률 합
- 예측 인덱스·클래스 이름과 Q1~Q4 답변

In [9]:
# 문제 3-2. 한 상품의 Softmax와 Argmax 복습하기
# 위 데이터와 작성 도움말을 보고 한 단계씩 작성하세요.
# 함수는 문제에 제공된 교안의 코드를 참고하세요.

# 1. 문제 2-1의 한 벡터용 Softmax 함수를 그대로 작성하세요.
# 2. `z = np.array([1, 2, 1])`, `classes = np.array(["의류", "식품", "가전"])`를 만드세요.
# 3. `p = softmax(z)`를 계산하고 `np.round(p, 3)`과 `np.sum(p)`를 출력하세요.
# 4. `index = np.argmax(p)`를 계산하고 index와 `classes[index]`를 출력하세요.

# 1. 1차원 벡터용 Softmax 함수 작성
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

# 2. z와 classes 배열 생성
z = np.array([1, 2, 1])
classes = np.array(["의류", "식품", "가전"])

# 3. p = softmax(z) 계산 및 소수점 셋째 자리 반올림, 합계 출력
p = softmax(z)
print("3. Softmax 확률 및 합계:")
print("클래스별 확률 p (반올림):", np.round(p, 3))
print("확률의 합 np.sum(p):", np.sum(p))
print("-" * 45)

# 4. np.argmax(p)로 인덱스 구하고 index와 예측 클래스 이름 출력
index = np.argmax(p)
print("4. 최종 예측 결과:")
print("예측 인덱스 (index):", index)
print("예측 클래스 이름:", classes[index])


3. Softmax 확률 및 합계:
클래스별 확률 p (반올림): [0.212 0.576 0.212]
확률의 합 np.sum(p): 1.0
---------------------------------------------
4. 최종 예측 결과:
예측 인덱스 (index): 1
예측 클래스 이름: 식품


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 최댓값을 뺀 배열, 지수값, 지수값의 합은 각각 무엇인가요?**

- 답변: 
- 배열: [-1, 0, -1] | 지수값: [0.3679, 1, 0.3679] | 지수값의 합: 1.7358

**Q2. 의류·식품·가전의 확률과 확률 합은 각각 얼마인가요?**

- 답변: 
- 각 확률 [0.212, 0.576, 0.212] | 확률 합 = 1

**Q3. Argmax 결과와 최종 예측 클래스는 무엇인가요? 인덱스 1은 확률 100%라는 뜻인가요?**

- 답변: Argmax 결과는 가장 큰 값의 위치 값(인덱스 번호)이며, 인덱스1은 확률 100%가 아니라, 이 문제에서는 해당 위치의 확률 p = 0.576을 의미한다.

**Q4. 의류와 가전의 확률이 같은 이유는 무엇인가요?**

- 답변: Z값이 1로 같기 때문

---

## 실습 마무리

1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?
2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?
3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?
4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?
5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?
6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?

### 실습 마무리 답변 작성란

**1. 회귀 예측값, 분류 점수, 예측 확률, 예측 클래스는 어떻게 다른가요?**

- 답변: 회귀 예측값은 1500원처럼 수치를 예측한 결과
- 분류 점수는 0.8로 확률로 바꾸기 전 숫자를 의미
- 예측 확률은 그 점수에 Sigmoid(이진) / softmax(다중)을 적용한 값을 의미

**2. Sigmoid(0)은 얼마이며, 임계값 0.5에서는 어떤 클래스로 정하나요?**

- 답변: Sigmoid(0)은 임계값인 0.5, 임계값을 따로 설정해놓지 않았다면 클래스는 1로 분류

**3. 구매 확률이 p일 때 미구매 확률은 무엇인가요?**

- 답변: 구매와 미구매가 가능한 두 결과 전부이기 때문에 1-p
- A의 구매 확률이 약 0.69이면 미구매 확률은 약 0.31

**4. Softmax는 무엇을 더한 합으로 각 지수값을 나누나요?**

- 답변: 한 데이터의 모든 클래스 점수에 지수 계산을 적용한 값들의 합으로 나눔
- 최대값을 뺀 [0, -1, -2]라면 1, 0.36, 0.13을 더한 1.503을 의미

**5. 여러 고객의 확률에서 Argmax의 axis=1은 무엇을 뜻하나요?**

- 답변: 한 고객을 나타내는 한 행 안에서 클래스별 확률을 비교
- [0.09, 0.66, 0.24]면 두 번째가 확률이 가장 크므로 인덱스 1을 반환

**6. 임계값을 높이거나 가장 높은 확률을 고르면 반드시 더 정확한 예측인가요?**

- 답변: 보장하지 않음. 임계값은 구매로 판단하는 기준이고 Argmax는 가장 큰 값의 위치를 고르는 규칙, 예측이 맞았는지 여부는 확률값보다는 실제 비교